In [45]:
# ===============================
# 1. Load necessary packages
# ===============================

import pandas as pd
import numpy as np
import os
import glob

import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch



In [3]:
# ===============================
# 2. Define folder paths
# ===============================

base_path = r"./"   # adjust only if necessary

ca_path     = os.path.join(base_path, "current_account_quarterly")
debt_path   = os.path.join(base_path, "debt_gdp_quarterly")
gdp_path    = os.path.join(base_path, "gdp_growth_quarterly")
reer_path   = os.path.join(base_path, "reer_quarterly")
spread_path = os.path.join(base_path, "spreads_per_country")


# ===============================
# 3. Helper function to load + stack country files
# ===============================

def load_quarterly_folder(folder_path, value_name):
    
    all_files = glob.glob(os.path.join(folder_path, "*.csv"))
    df_list = []

    for file in all_files:
        country = os.path.basename(file)[:2]   # AT, BE, DE ...
        
        df = pd.read_csv(file)
        df.columns = ["quarter", value_name]
        df["country"] = country
        
        df_list.append(df)

    df_final = pd.concat(df_list, ignore_index=True)
    return df_final

In [4]:
# ===============================
# 4. Load datasets
# ===============================

ca     = load_quarterly_folder(ca_path, "current_account")
debt   = load_quarterly_folder(debt_path, "debt_gdp")
gdp    = load_quarterly_folder(gdp_path, "gdp_growth")
reer   = load_quarterly_folder(reer_path, "reer")
spread = load_quarterly_folder(spread_path, "spread")

In [5]:
# ===============================
# 5. Restrict timespan 2004–2024
# ===============================

def filter_timespan(df):
    df["year"] = df["quarter"].str[:4].astype(int)
    df = df[(df["year"] >= 2004) & (df["year"] <= 2024)]
    return df.drop(columns="year")

ca     = filter_timespan(ca)
debt   = filter_timespan(debt)
gdp    = filter_timespan(gdp)
reer   = filter_timespan(reer)
spread = filter_timespan(spread)

In [6]:
# ===============================
# 6. Merge all datasets
# ===============================

df = spread.merge(debt,  on=["country", "quarter"], how="inner")
df = df.merge(ca,        on=["country", "quarter"], how="inner")
df = df.merge(gdp,       on=["country", "quarter"], how="inner")
df = df.merge(reer,      on=["country", "quarter"], how="inner")

df = df.sort_values(["country", "quarter"]).reset_index(drop=True)

df.head()

,quarter,spread,country,debt_gdp,current_account,gdp_growth,reer
0,2004Q1,0.115590,AT,69.3,3354.0,0.001946,100.136667
1,2004Q2,0.121980,AT,72.1,623.0,0.010275,99.240000
2,2004Q3,0.072588,AT,72.1,254.0,0.011409,99.493333
3,2004Q4,0.064820,AT,65.9,800.0,0.003500,100.460000
4,2005Q1,0.002949,AT,74.4,3499.0,0.002270,100.376667


In [7]:
# ===============================
# 7. Check missing values
# ===============================

print(df.isna().sum())
print("Observations:", len(df))
print("Countries:", df["country"].nunique())

quarter            0
spread             0
country            0
debt_gdp           0
current_account    0
gdp_growth         0
reer               0
dtype: int64
Observations: 924
Countries: 11


In [9]:
# ===============================
# Drop Germany (DE)
# ===============================

df = df[df["country"] != "DE"].reset_index(drop=True)

print("Countries in dataset:")
print(sorted(df["country"].unique()))

print("Number of countries:", df["country"].nunique())

Countries in dataset:
['AT', 'BE', 'ES', 'FI', 'FR', 'GR', 'IE', 'IT', 'LU', 'NL', 'PT']
Number of countries: 11


In [19]:
df

,quarter,spread,country,debt_gdp,current_account,gdp_growth,reer
0,2004Q1,0.115590,AT,69.3,3354.0,0.001946,100.136667
1,2004Q2,0.121980,AT,72.1,623.0,0.010275,99.240000
2,2004Q3,0.072588,AT,72.1,254.0,0.011409,99.493333
3,2004Q4,0.064820,AT,65.9,800.0,0.003500,100.460000
4,2005Q1,0.002949,AT,74.4,3499.0,0.002270,100.376667
...,...,...,...,...,...,...,...
919,2023Q4,0.741116,PT,96.9,-1133.0,0.007326,98.210000
920,2024Q1,0.734683,PT,98.1,1864.0,0.005805,97.930000
921,2024Q2,0.676747,PT,99.1,996.0,0.005587,98.773333
922,2024Q3,0.642290,PT,95.9,3395.0,0.001933,98.236667


In [20]:
# ===============================
# Load GDP level
# ===============================

gdp_level_path = os.path.join(base_path, "gdp_quarterly")
gdp_level = load_quarterly_folder(gdp_level_path, "gdp_level")

gdp_level = filter_timespan(gdp_level)

In [21]:
df = df.merge(gdp_level, on=["country", "quarter"], how="inner")

In [33]:
df

,quarter,spread,country,debt_gdp,current_account,gdp_growth,reer,gdp_level,ca_gdp
0,2004Q1,0.115590,AT,69.3,3354.0,0.001946,100.136667,66238.8,0.050635
1,2004Q2,0.121980,AT,72.1,623.0,0.010275,99.240000,66922.9,0.009309
2,2004Q3,0.072588,AT,72.1,254.0,0.011409,99.493333,67690.8,0.003752
3,2004Q4,0.064820,AT,65.9,800.0,0.003500,100.460000,67928.1,0.011777
4,2005Q1,0.002949,AT,74.4,3499.0,0.002270,100.376667,68082.5,0.051394
...,...,...,...,...,...,...,...,...,...
919,2023Q4,0.741116,PT,96.9,-1133.0,0.007326,98.210000,51565.2,-0.021972
920,2024Q1,0.734683,PT,98.1,1864.0,0.005805,97.930000,51865.4,0.035939
921,2024Q2,0.676747,PT,99.1,996.0,0.005587,98.773333,52156.0,0.019097
922,2024Q3,0.642290,PT,95.9,3395.0,0.001933,98.236667,52256.9,0.064967


In [22]:
# ===============================
# Construct CA/GDP ratio
# ===============================

df["ca_gdp"] = df["current_account"] / df["gdp_level"]

In [23]:
# ===============================
# 8. OLS Regression
# ===============================

X = df[["debt_gdp", "current_account", "gdp_growth", "reer"]]
X = sm.add_constant(X)

y = df["spread"]

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 spread   R-squared:                       0.326
Model:                            OLS   Adj. R-squared:                  0.323
Method:                 Least Squares   F-statistic:                     111.3
Date:                Tue, 03 Mar 2026   Prob (F-statistic):           2.24e-77
Time:                        23:44:27   Log-Likelihood:                -1883.2
No. Observations:                 924   AIC:                             3776.
Df Residuals:                     919   BIC:                             3800.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const             -12.6367      1.298     

In [24]:
model_quad = smf.ols(
    "spread ~ debt_gdp + I(debt_gdp**2) + ca_gdp + gdp_growth + reer",
    data=df
).fit(cov_type="HC1")

print(model_quad.summary())

                            OLS Regression Results                            
Dep. Variable:                 spread   R-squared:                       0.365
Model:                            OLS   Adj. R-squared:                  0.361
Method:                 Least Squares   F-statistic:                     40.18
Date:                Tue, 03 Mar 2026   Prob (F-statistic):           2.03e-37
Time:                        23:44:59   Log-Likelihood:                -1855.9
No. Observations:                 924   AIC:                             3724.
Df Residuals:                     918   BIC:                             3753.
Df Model:                           5                                         
Covariance Type:                  HC1                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept          -11.2722      2.048  

In [25]:
b1 = model_quad.params["debt_gdp"]
b2 = model_quad.params["I(debt_gdp ** 2)"]

turning_point = -b1 / (2 * b2)
print("Debt turning point:", turning_point)

Debt turning point: 16.765607411979396


In [36]:
exog = df_panel[["debt_gdp", "ca_gdp", "gdp_growth", "reer"]]

In [41]:

df["quarter"] = pd.PeriodIndex(df["quarter"].astype(str), freq="Q").to_timestamp()

# --- Set MultiIndex ---
df_panel = df.set_index(["country", "quarter"]).sort_index()

# --- Ensure numeric ---
vars_needed = ["spread", "debt_gdp", "ca_gdp", "gdp_growth", "reer"]
df_panel[vars_needed] = df_panel[vars_needed].apply(pd.to_numeric, errors="coerce")
df_panel = df_panel.dropna()

# --- Define regressors (NO constant) ---
exog = df_panel[["debt_gdp", "ca_gdp", "gdp_growth", "reer"]]

# --- Estimate FE ---
model_fe = PanelOLS(
    df_panel["spread"],
    exog,
    entity_effects=True
)

results_fe = model_fe.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(results_fe)

                          PanelOLS Estimation Summary                           
Dep. Variable:                 spread   R-squared:                        0.1916
Estimator:                   PanelOLS   R-squared (Between):             -100.95
No. Observations:                 924   R-squared (Within):               0.1916
Date:                Tue, Mar 03 2026   R-squared (Overall):             -44.502
Time:                        23:57:33   Log-likelihood                   -1802.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      53.860
Entities:                          11   P-value                           0.0000
Avg Obs:                       84.000   Distribution:                   F(4,909)
Min Obs:                       84.000                                           
Max Obs:                       84.000   F-statistic (robust):             11.280
                            

In [42]:
print(results_fe.summary.as_text())

                          PanelOLS Estimation Summary                           
Dep. Variable:                 spread   R-squared:                        0.1916
Estimator:                   PanelOLS   R-squared (Between):             -100.95
No. Observations:                 924   R-squared (Within):               0.1916
Date:                Tue, Mar 03 2026   R-squared (Overall):             -44.502
Time:                        23:57:33   Log-likelihood                   -1802.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      53.860
Entities:                          11   P-value                           0.0000
Avg Obs:                       84.000   Distribution:                   F(4,909)
Min Obs:                       84.000                                           
Max Obs:                       84.000   F-statistic (robust):             11.280
                            

In [43]:
print(results_fe.params)

debt_gdp       0.044153
ca_gdp         1.660034
gdp_growth   -10.757110
reer           0.140843
Name: parameter, dtype: float64


In [ ]:


file_path = "regression_results.pdf"
doc = SimpleDocTemplate(file_path)
elements = []
styles = getSampleStyleSheet()

def regression_to_table(results, title):
    elements.append(Paragraph(title, styles['Heading2']))
    elements.append(Spacer(1, 0.2 * inch))
    
    df_table = pd.DataFrame({
        "Coefficient": results.params,
        "Std. Error": getattr(results, "std_errors", getattr(results, "bse", None)),
        "t-stat": getattr(results, "tstats", getattr(results, "tvalues", None)),
        "p-value": results.pvalues
    })
    
    df_table = df_table.round(4)
    df_table = df_table.reset_index().rename(columns={"index": "Variable"})
    
    data = [df_table.columns.tolist()] + df_table.values.tolist()
    
    table = Table(data, repeatRows=1)
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('GRID', (0,0), (-1,-1), 0.25, colors.grey),
        ('FONTSIZE', (0,0), (-1,-1), 8),
        ('LEFTPADDING', (0,0), (-1,-1), 4),
        ('RIGHTPADDING', (0,0), (-1,-1), 4),
    ]))
    
    elements.append(table)
    elements.append(PageBreak())

# Add your regressions
regression_to_table(model, "Pooled OLS Results")
regression_to_table(model_quad, "Pooled OLS with Debt^2 Results")
regression_to_table(results_fe, "Country Fixed Effects Results")

doc.build(elements)

print("PDF saved as:", file_path)

PDF saved as: regression_results.pdf
